In [1]:
from pyflink.table import EnvironmentSettings, TableEnvironment, DataTypes
from pyflink.table.expressions import col, call_sql
from pyflink.common import Configuration
import os

In [2]:
config = Configuration()
config.set_string("jobmanager.rpc.address", "jobmanager")
config.set_string("rest.address", "jobmanager")
config.set_string("rest.port", "8081")
config.set_string("execution.target", "remote")
config.set_string("table.exec.sink.type", "unified")
config.set_string("table.exec.resource.default-parallelism", "5")

env_settings = EnvironmentSettings.new_instance().in_streaming_mode().with_configuration(config).build()
t_env = TableEnvironment.create(env_settings)

In [3]:
pg_user = os.getenv("POSTGRES_USER", "postgres")
pg_password = os.getenv("POSTGRES_PASSWORD", "postgres")
pg_host = os.getenv("POSTGRES_HOST", "postgres")
pg_port = str(os.getenv("POSTGRES_PORT", "5432"))
pg_db = "postgres"

jdbc_url = f"jdbc:postgresql://{pg_host}:{pg_port}/{pg_db}"

In [4]:
t_env.drop_table('kafka_source')

False

In [5]:
source_kafka_query = '''
    CREATE TABLE kafka_source (
        id BIGINT NOT NULL,
        customer_first_name varchar(50),
        customer_last_name varchar(50),
        customer_age INT NOT NULL,
        customer_email varchar(50),
        customer_country varchar(50),
        customer_postal_code varchar(50),
        customer_pet_type varchar(50),
        customer_pet_name varchar(50),
        customer_pet_breed varchar(50),
        seller_first_name varchar(50),
        seller_last_name varchar(50),
        seller_email varchar(50),
        seller_country varchar(50),
        seller_postal_code varchar(50),
        product_name varchar(50),
        product_category varchar(50),
        product_price FLOAT NOT NULL,
        product_quantity INT NOT NULL,
        sale_date varchar(50),
        sale_customer_id BIGINT NOT NULL,
        sale_seller_id BIGINT NOT NULL,
        sale_product_id BIGINT NOT NULL,
        sale_supplier_id BIGINT NOT NULL,
        sale_store_id BIGINT NOT NULL,
        sale_quantity INT NOT NULL,
        sale_total_price FLOAT NOT NULL,
        store_name varchar(50),
        store_location varchar(50),
        store_city varchar(50),
        store_state varchar(50),
        store_country varchar(50),
        store_phone varchar(50),
        store_email varchar(50),
        pet_category varchar(50),
        product_weight FLOAT NOT NULL,
        product_color varchar(50),
        product_size varchar(50),
        product_brand varchar(50),
        product_material varchar(50),
        product_description varchar(1024),
        product_rating FLOAT NOT NULL,
        product_reviews INT NOT NULL,
        product_release_date varchar(50),
        product_expiry_date varchar(50),
        supplier_name varchar(50),
        supplier_contact varchar(50),
        supplier_email varchar(50),
        supplier_phone varchar(50),
        supplier_address varchar(50),
        supplier_city varchar(50),
        supplier_country varchar(50)
    ) WITH (
        'connector' = 'kafka',
        'topic' = 'data-topic',
        'properties.bootstrap.servers' = 'kafka:9092',
        'properties.group.id' = 'flink_consumer',
        'scan.startup.mode' = 'earliest-offset',
        'scan.bounded.mode' = 'latest-offset',
        'format' = 'json'
    )
'''

t_env.execute_sql(source_kafka_query).print()

OK


In [6]:
def register_sink(table_name, schema):
    t_env.execute_sql(f"""
        CREATE TABLE {table_name} ({schema})
        WITH (
            'connector' = 'jdbc',
            'url' = '{jdbc_url}',
            'table-name' = '{table_name}',
            'username' = '{pg_user}',
            'password' = '{pg_password}',
            'sink.max-retries' = '5',
            'sink.buffer-flush.interval' = '2s'
        )
    """)

def gen_id(column_name):
    return call_sql(f"ABS(MOD(HASH_CODE({column_name}), 2147483647))")

In [7]:
register_sink('dim_countries', 'id INT PRIMARY KEY NOT ENFORCED, name STRING')
register_sink('dim_pet_categories', 'id INT PRIMARY KEY NOT ENFORCED, category STRING')

register_sink('dim_customers', '''
    id INT PRIMARY KEY NOT ENFORCED, customer_first_name STRING, customer_last_name STRING,
    customer_age INT, customer_email STRING, customer_postal_code STRING,
    customer_pet_type STRING, customer_pet_name STRING, customer_pet_breed STRING,
    pet_category STRING, pet_category_id INT, country_id INT
''')

register_sink('dim_suppliers', '''
    id INT PRIMARY KEY NOT ENFORCED, supplier_name STRING, supplier_contact STRING,
    supplier_email STRING, supplier_phone STRING, supplier_address STRING,
    supplier_city STRING, country_id INT
''')

register_sink('dim_stores', '''
    id INT PRIMARY KEY NOT ENFORCED, store_name STRING, store_location STRING,
    store_city STRING, store_state STRING, store_phone STRING, store_email STRING, country_id INT
''')

register_sink('dim_sellers', '''
    id INT PRIMARY KEY NOT ENFORCED, seller_first_name STRING, seller_last_name STRING,
    seller_email STRING, seller_postal_code STRING, seller_store_id INT, country_id INT
''')

register_sink('dim_products', '''
    id INT PRIMARY KEY NOT ENFORCED, product_name STRING, product_category STRING, product_price FLOAT,
    product_quantity INT, product_weight FLOAT, product_color STRING, product_size STRING,
    product_brand STRING, product_material STRING, product_description STRING,
    product_rating FLOAT, product_reviews INT, product_release_date STRING, product_expiry_date STRING,
    product_supplier_id INT
''')

register_sink('fact_sales', '''
    id INT PRIMARY KEY NOT ENFORCED, sale_date STRING, sale_customer_id INT, sale_seller_id INT,
    sale_product_id INT, sale_store_id INT, sale_supplier_id INT,
    sale_quantity INT, sale_total_price FLOAT
''')

In [8]:
df = t_env.from_path("kafka_source")

countries = df.select(col("customer_country").alias("name")).where(col("name").is_not_null) \
    .union_all(df.select(col("seller_country").alias("name"))) \
    .union_all(df.select(col("store_country").alias("name"))) \
    .union_all(df.select(col("supplier_country").alias("name"))) \
    .distinct() \
    .select(
        gen_id("name").alias("id"),
        col("name")
    )

pet_cats = df.select(col("pet_category").alias("category")).where(col("category").is_not_null) \
    .distinct() \
    .select(
        gen_id("category").alias("id"),
        col("category")
    )

customers = df.select(
    col("id").cast(DataTypes.INT()),
    col("customer_first_name"),
    col("customer_last_name"),
    col("customer_age"),
    col("customer_email"),
    col("customer_postal_code"),
    col("customer_pet_type"),
    col("customer_pet_name"),
    col("customer_pet_breed"),
    col("pet_category"),
    gen_id("pet_category").alias("pet_cat_id"),
    gen_id("customer_country").alias("c_id")
)

stores = df.select(
    col("id").cast(DataTypes.INT()),
    col("store_name"),
    col("store_location"),
    col("store_city"),
    col("store_state"),
    col("store_phone"),
    col("store_email"),
    gen_id("store_country").alias("country_id")
)

suppliers = df.select(
    col("id").cast(DataTypes.INT()),
    col("supplier_name"),
    col("supplier_contact"),
    col("supplier_email"),
    col("supplier_phone"),
    col("supplier_address"),
    col("supplier_city"),
    gen_id("supplier_country").alias("country_id")
)

sellers = df.select(
    col("id").cast(DataTypes.INT()),
    col("seller_first_name"),
    col("seller_last_name"),
    col("seller_email"),
    col("seller_postal_code"),
    col("sale_store_id").cast(DataTypes.INT()).alias("seller_store_id"),
    gen_id("seller_country").alias("country_id")
)

products = df.select(
    col("id").cast(DataTypes.INT()),
    col("product_name"),
    col("product_category"),
    col("product_price"),
    col("product_quantity"),
    col("product_weight"),
    col("product_color"),
    col("product_size"),
    col("product_brand"),
    col("product_material"),
    col("product_description"),
    col("product_rating"),
    col("product_reviews"),
    col("product_release_date"),
    col("product_expiry_date"),
    col("sale_supplier_id").cast(DataTypes.INT()).alias("product_supplier_id")
)

sales = df.select(
    col("id").cast(DataTypes.INT()),
    col("sale_date"),
    col("sale_customer_id").cast(DataTypes.INT()),
    col("sale_seller_id").cast(DataTypes.INT()),
    col("sale_product_id").cast(DataTypes.INT()),
    col("sale_store_id").cast(DataTypes.INT()),
    col("sale_supplier_id").cast(DataTypes.INT()),
    col("sale_quantity"),
    col("sale_total_price")
)

In [9]:
statement_set = t_env.create_statement_set()
statement_set.add_insert("dim_countries", countries)
statement_set.add_insert("dim_pet_categories", pet_cats)
statement_set.add_insert("dim_suppliers", suppliers)
statement_set.add_insert("dim_stores", stores)
statement_set.add_insert("dim_customers", customers)
statement_set.add_insert("dim_sellers", sellers)
statement_set.add_insert("dim_products", products)
statement_set.add_insert("fact_sales", sales)

statement_set.execute()